# Activity 6: MCP, the Standard Plug for Tools

**Week 6 Day 3 · Making your tools work in something other than your own script**

In Activity 4 you gave a model three tools and it used them. That worked, but look at what you actually had to build to make it work: a hand-written JSON schema for every function, a dictionary mapping names back to Python functions, and a loop. All of it lived inside one notebook.

Now suppose a teammate wants those same claim tools in *their* agent. They copy your schemas. Suppose you want them available inside Claude Code, or Cursor, or any other AI client. You cannot, there is no way to hand your notebook's dictionary to somebody else's application.

**MCP (Model Context Protocol) is the fix.** You run your tools as a small standalone program that speaks one agreed-upon protocol, and then *any* client that also speaks it can discover and call your tools without you writing a line of integration code. USB solved this for hardware. MCP solves it for model tools.

## What you will learn

- Why hand-written tool schemas do not travel between projects
- How to expose Python functions as an MCP server with almost no ceremony
- How a client *discovers* tools it was never told about, at runtime
- How MCP tools plug into the exact same OpenAI loop you wrote in Activity 4
- What is actually happening when Claude Code uses a tool

---
## Setup

You installed `mcp` in [Activity 0](./Activity_0_Environment_and_API_Setup.md). Everything here runs locally except the model calls in Section 5, which reuse your `OPENAI_API_KEY`.

In [ ]:
import json
import os
import sys

from dotenv import load_dotenv
from mcp import Client, StdioServerParameters, stdio_client
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("ready")

---
# 1. Write the server

An MCP server is a normal Python program. `%%writefile` saves the cell below to a real file next to this notebook, because a server has to be a file on disk: the client will launch it as a separate process in a moment.

These are the same three claim tools from Activity 4, unchanged.

In [ ]:
%%writefile claims_mcp_server.py
"""A tiny MCP server exposing the claim tools from Activity 4."""

from mcp.server import MCPServer

mcp = MCPServer("claims-tools")

CLAIMS_DB = {
    "CLM_101": {"status": "Approved", "amount": 3400.0, "type": "Auto Collision"},
    "CLM_102": {"status": "Approved", "amount": 1250.0, "type": "Property Loss"},
    "CLM_103": {"status": "Denied", "amount": 0.0, "type": "Fraud Flag"},
}

POLICIES_DB = {
    "POL_991": {"deductible": 500.0, "coverage": "Full Comprehensive"},
    "POL_992": {"deductible": 1000.0, "coverage": "Liability Only"},
}


@mcp.tool()
def get_claim_status(claim_id: str) -> dict:
    """Look up an insurance claim's status, amount, and type by claim ID."""
    return CLAIMS_DB.get(claim_id, {"error": f"{claim_id} not found"})


@mcp.tool()
def get_policy_deductible(policy_id: str) -> dict:
    """Look up a policy's deductible amount and coverage type by policy ID."""
    return POLICIES_DB.get(policy_id, {"error": f"{policy_id} not found"})


@mcp.tool()
def calculate_net_payout(claim_amount: float, deductible: float) -> dict:
    """Subtract a deductible from a claim amount to get the net payout."""
    return {"net_payout": round(max(0.0, claim_amount - deductible), 2)}


if __name__ == "__main__":
    mcp.run()

Read that file and notice what is **missing**: there is no JSON schema anywhere. In Activity 4 you wrote about 30 lines of nested `"parameters": {"type": "object", "properties": ...}` by hand. Here `@mcp.tool()` builds all of that itself by reading the function's type hints and its docstring. That is why the type hints and docstrings are not decoration in an MCP server, they are the interface. A vague docstring produces a tool the model uses badly.

`mcp.run()` at the bottom starts the server on **stdio**, meaning it talks over standard input and output, the same pipes a shell uses. The client will start this file as a subprocess and speak to it through those pipes. No ports, no HTTP, no network.

---
# 2. Discover the tools

`StdioServerParameters` describes *how to launch* the server: run this Python interpreter, on this file. `Client` starts it, performs the protocol handshake, and gives you a session.

The important part is `list_tools()`. Your notebook has never imported the server, never seen `CLAIMS_DB`, and was never told what tools exist. It asks, at runtime, and the server answers.

In [ ]:
server_params = StdioServerParameters(command=sys.executable, args=["claims_mcp_server.py"])

async with Client(stdio_client(server_params)) as session:
    listed = await session.list_tools()

for tool in listed.tools:
    print(tool.name, "->", tool.description)

Three tools discovered, with the descriptions taken straight from the docstrings.

Two things about the code shape. First, `await` works directly in a notebook cell, no `asyncio.run(...)` needed. Second, **every cell that talks to the server opens its own `async with` block.** A connection cannot be opened in one cell and reused in the next, because the notebook runs each cell as a separate task and the connection has to be opened and closed inside the same one. So each cell below starts the server, does its work, and shuts it down.

Now look at the schema the server generated for one of those tools.

In [ ]:
print(json.dumps(listed.tools[0].input_schema, indent=2))

Compare that to the `tools` list you hand-wrote in Activity 4 Section 2. Same `type`, same `properties`, same `required`. It is the same JSON Schema, describing the same function, in the same shape.

The difference is who wrote it. In Activity 4, you did, and it only existed inside that one notebook. Here the server generated it from the Python function and handed it over the wire to a client that had never seen the code.

---
# 3. Call a tool by hand

Same as Activity 4 Section 4: before letting a loop do it, do one call yourself and look at what comes back.

In [ ]:
async with Client(stdio_client(server_params)) as session:
    result = await session.call_tool("get_claim_status", {"claim_id": "CLM_101"})

print(result.content[0].text)

The result arrives as text content, because the protocol has to carry results between two separate processes that share no Python objects. Your function returned a `dict`; MCP serialized it to JSON to send it across, and `result.content[0].text` is that JSON.

That is the only real difference from Activity 4. There, `AVAILABLE_FUNCTIONS[name](**args)` called a function sitting in the same notebook. Here the call crossed a process boundary and came back as text. Everything else about the pattern, ask for a tool by name with arguments and get a result, is identical.

---
# 4. Bridge MCP tools into the OpenAI loop

The model still does not know MCP exists. It only understands the `tools=[...]` format from Activity 4. So translate: an MCP tool has a `name`, a `description`, and an `input_schema`, and an OpenAI tool needs a `name`, a `description`, and `parameters`. That is the whole conversion.

In [ ]:
def mcp_to_openai(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.input_schema,
        },
    }


openai_tools = [mcp_to_openai(t) for t in listed.tools]
print(json.dumps(openai_tools[0], indent=2))

Three keys moved to three other keys. Nothing else happened, and nothing was hand-written.

This is the point of the whole notebook: **MCP did not replace function calling.** The model still stops with `finish_reason == "tool_calls"`, still asks for a name and arguments, still needs you to run it and hand the result back. MCP replaced only the part where *you* had to write the schemas and hard-code the function lookup. Discovery and transport are standardized; the engine underneath is the loop you already built.

---
# 5. The same loop, MCP-backed

This is Activity 4's `run_conversation`, with two lines changed. The tools come from `list_tools()` instead of a hand-written list, and executing one is `await session.call_tool(...)` instead of `AVAILABLE_FUNCTIONS[name](**args)`.

The whole loop lives inside one `async with` block so the server stays running for every round.

In [ ]:
async def run_mcp_conversation(prompt, max_rounds=5):
    async with Client(stdio_client(server_params)) as session:
        tools = [mcp_to_openai(t) for t in (await session.list_tools()).tools]
        messages = [{"role": "user", "content": prompt}]

        for round_num in range(1, max_rounds + 1):
            response = client.chat.completions.create(
                model="gpt-4o-mini", messages=messages, tools=tools
            )
            reply = response.choices[0].message

            if response.choices[0].finish_reason != "tool_calls":
                print(f"[round {round_num}] model answered, stopping")
                return reply.content

            messages.append(reply)
            for call in reply.tool_calls:
                args = json.loads(call.function.arguments)
                result = await session.call_tool(call.function.name, args)
                text = result.content[0].text
                print(f"[round {round_num}] ran {call.function.name}({args}) -> {json.loads(text)}")
                messages.append({"role": "tool", "tool_call_id": call.id, "content": text})

        return "Gave up after max_rounds without a final answer."

In [ ]:
answer = await run_mcp_conversation(
    "Claim CLM_101 is on policy POL_991. What is the net payout after the deductible?"
)
print("\nFinal answer:", answer)

Same question as Activity 4, same multi-round behavior, same final answer. The model cannot tell the difference, and that is exactly right: from the model's side nothing changed at all.

What changed is on your side. The notebook no longer contains the tools. It contains an address for finding them.

---
# 6. What this buys you

| | Activity 4 | This notebook |
|---|---|---|
| **Who writes the tool schema** | You, by hand, per tool | Generated from type hints |
| **Where the tools live** | Inside the notebook | A separate program |
| **How the client learns about them** | Hard-coded in the script | Asked for at runtime |
| **Adding a fourth tool** | Edit the schema list and the function dict | Add a function to the server, restart it |
| **Usable by another application** | No | Yes, any MCP client |

That last row is the one that matters, and it is why you already use MCP without knowing it.

When Claude Code reads a file, runs a command, or searches your repo, it is doing exactly what you just did: it launched a server, called `list_tools()` to find out what that server can do, decided which tool it needed, called it, and fed the result back into its own conversation loop. Same protocol, same handshake, same round trip. The reason Claude Code can talk to GitHub, or a database, or your own `claims_mcp_server.py`, is that none of those integrations are built into Claude Code. They are separate servers that speak the protocol.

So the honest summary of this whole day: a chat assistant that seems to "just know how to do things" is a model, in a loop, calling tools somebody wrote and registered. You have now written every one of those pieces yourself.

---
# Your Turn

Work in your own copy under `student-work/week6/day3/`.

1. Add a fourth tool to `claims_mcp_server.py`: `list_claims_by_status(status: str) -> list`, returning every claim in `CLAIMS_DB` matching that status. Re-run the `%%writefile` cell, then re-run the discovery cell in Section 2. Confirm it appears with no other change to your client code, and note how many places you had to edit compared with adding a tool in Activity 4.
2. Ask `run_mcp_conversation` a question that needs your new tool, for example *"Which claims were denied?"*

**Stretch goal:** deliberately weaken a docstring in the server, change `get_policy_deductible`'s to just `"Gets data."`, restart, and re-run the multi-step question from Section 5. Does the model still pick the right tool? This is the most useful thing you can learn about MCP: the docstring is not a comment, it is the only thing the model has to go on.

## What you did

- Exposed plain Python functions as an MCP server with no hand-written schemas.
- Discovered tools at runtime from a client that had never seen the server's code.
- Called a tool across a process boundary and read the result back.
- Translated MCP tool definitions into the OpenAI `tools=` format in four lines.
- Ran Activity 4's loop unchanged against MCP-backed tools.
- Connected all of it to what Claude Code is actually doing when it uses a tool.